# 内置Middleware - TodoListMiddleware
- 作用：规划复杂任务，列出Todo List，指定模型按步完成
- 场景方案选择：
    - 复杂但步骤固定的任务：使用LangGraph
    - 复杂但步骤多变，且会面临失败，需要反复调整：使用TodoListMiddleware
    - 简单任务则不需要，浪费算力
- 参数：
    - system_prompt: 生成Todo List的提示词
    - tool_description: 自定义write_tools工具描述信息

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from rich import print as rprint
from langchain_core.tools import tool

from common import init_simple_dashscope_model

model = init_simple_dashscope_model('qwen-max')


In [6]:
from langchain.tools import tool
from pathlib import Path
import subprocess

WORKSPACE = Path("../todo_workspace")

@tool
def list_files(path: str = ".") -> str:
    """
    列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
        path: 工作区下的相对路径，一定指向目录，默认为.，表示工作区根路径，不能访问工作区外的目录
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()

    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"

    if not target.exists():
        return f"错误：目录不存在: {path}"

    if not target.is_dir():
        return f"错误：不是目录: {path}"

    items = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    if not items:
        return f"目录为空: {path}"

    lines = []
    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")

    return "\n".join(lines)

@tool
def read_file(path: str) -> str:
    """
    读取工作区中的文本文件内容。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许读取工作区内的文件。"
    if not file_path.exists():
        return f"错误：文件不存在: {path}"
    return file_path.read_text(encoding="utf-8")


@tool
def write_file(path: str, content: str) -> str:
    """
    写入工作区中的文本文件。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
        content: 写入文件的内容
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许写入工作区内的文件。"
    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件: {path}"


@tool
def run_tests() -> str:
    """
    在工作区运行 pytest -q，并返回输出。
    不接收任何参数，返回格式为
    returncode=0|1
    STDOUT:
    STDERR:
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"运行测试失败: {e}"


In [7]:
from langchain.agents.middleware import TodoListMiddleware
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from rich import print as rprint
agent = create_agent(
    model=model,
    tools=[list_files, read_file, write_file,run_tests],
    middleware=[TodoListMiddleware()],
    system_prompt="你是一个代码修复助手。遇到多步骤任务时，先使用 write_todos 制定待办事项；"
        "然后读取文件、修复代码并运行测试。工作全部在工作区下进行。"
)


response = agent.invoke({
    "messages" : [HumanMessage("请测试并修复工作区下的my_add.py文件中的代码")]
})


rprint(response)

{
    'messages': [
        HumanMessage(
            content='请测试并修复工作区下的my_add.py文件中的代码',
            additional_kwargs={},
            response_metadata={},
            id='ccb78af7-6569-47a1-9c04-24399525351e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 144,
                    'prompt_tokens': 1690,
                    'total_tokens': 1834,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 1024
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-3bd99a73-c3de-9da2-be6c-7371c088bd43',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f8ccd-e2a5-7c73-a88b-a98569b323e5-0',
            tool_calls=[
                {
                    'name': 'write_todos',
                    'args': {
                        'todos': [
                            {'content': '列出工作区根目录下的文件，找到my_add.py', 'status': 'pending'},
                            {'content': '读取my_add.py文件的内容', 'status': 'pending'},
                            {'content': '检查my_add.py中的代码是否有明显错误', 'status': 'pending'},
                            {'content': '如果存在错误，则修复这些错误', 'status': 'pending'},
                            {'content': '运行测试用例以确保my_add.py中的函数正确性', 'status': 'pending'},
                            {
                                'content': '如果测试失败，则根据失败信息进一步调试和修复my_add.py',
                                'status': 'pending'
                            }
                        ]
                    },
                    'id': 'call_dfa539c61de54428b42c82',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1690,
                'output_tokens': 144,
                'total_tokens': 1834,
                'input_token_details': {'cache_read': 1024},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Updated todo list to [{'content': '列出工作区根目录下的文件，找到my_add.py', 'status': 
'pending'}, {'content': '读取my_add.py文件的内容', 'status': 'pending'}, {'content': 
'检查my_add.py中的代码是否有明显错误', 'status': 'pending'}, {'content': '如果存在错误，则修复这些错误', 'status': 
'pending'}, {'content': '运行测试用例以确保my_add.py中的函数正确性', 'status': 'pending'}, {'content': 
'如果测试失败，则根据失败信息进一步调试和修复my_add.py', 'status': 'pending'}]",
            name='write_todos',
            id='46ead70e-5ee9-4c40-8948-ef5c8ae08def',
            tool_call_id='call_dfa539c61de54428b42c82'
        ),
        AIMessage(
            content='我已经创建了一个待办事项列表来帮助我们有条不紊地测试和修复 `my_add.py` 
文件。现在，我将开始执行第一个任务，即列出工作区根目录下的文件以找到 `my_add.py`。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 72,
                    'prompt_tokens': 1974,
                    'total_tokens': 2046,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-92970050-e7d5-95dd-b001-a13c81bf2339',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f8ccd-f8c7-7cc